# Bước 4 — Mô hình MRC sinh câu trả lời (generative) trên ViMMRC 2.0

Tải dữ liệu ViMMRC 2.0, fine-tune mô hình seq2seq tiếng Việt (ViT5) nhận đầu vào `(passage, câu hỏi)` và **sinh ra câu trả lời dạng văn bản tự do** (không chọn trắc nghiệm A/B/C/D).

In [1]:
!pip install -q transformers datasets accelerate huggingface_hub sentencepiece evaluate rouge_score

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## 1. Tải dữ liệu

In [2]:
import json
from huggingface_hub import hf_hub_download

REPO = "uitnlp/vimmrc2.0"

def load_split(name):
    path = hf_hub_download(repo_id=REPO, filename=f"{name}_vimmrc.json", repo_type="dataset")
    with open(path, encoding="utf-8") as f:
        return json.load(f)

train_raw = load_split("train")
dev_raw = load_split("dev")
test_raw = load_split("test")
len(train_raw), len(dev_raw), len(test_raw)

train_vimmrc.json:   0%|          | 0.00/3.33M [00:00<?, ?B/s]

C:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\datasets--uitnlp--vimmrc2.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


dev_vimmrc.json:   0%|          | 0.00/523k [00:00<?, ?B/s]

test_vimmrc.json:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

(482, 70, 147)

## 2. Chuẩn hoá thành ví dụ sinh văn bản

ViMMRC là dữ liệu trắc nghiệm (4 `options`, đáp án là chữ A-D trong `answers`). Để có nhãn seq2seq, ta lấy **văn bản của phương án đúng** làm câu trả lời mục tiêu — mô hình học sinh ra đúng câu trả lời đó dưới dạng tự do, không cần biết trước danh sách lựa chọn khi suy luận.

In [3]:
LETTER2IDX = {"A": 0, "B": 1, "C": 2, "D": 3}

def flatten(raw):
    examples = []
    for article in raw:
        passage = article["article"]
        for q, opts, ans in zip(article["questions"], article["options"], article["answers"]):
            if ans not in LETTER2IDX or len(opts) != 4:
                continue  # bỏ qua câu hỏi thiếu/lỗi định dạng
            examples.append({
                "passage": passage,
                "question": q,
                "answer_text": opts[LETTER2IDX[ans]],
            })
    return examples

train_examples = flatten(train_raw)
dev_examples = flatten(dev_raw)
test_examples = flatten(test_raw)
len(train_examples), len(dev_examples), len(test_examples)

(3600, 564, 1109)

## 3. Tokenize (seq2seq)

In [4]:
from transformers import AutoTokenizer

MODEL_NAME = "VietAI/vit5-base"  # seq2seq (T5) tiếng Việt; đổi sang "vinai/bartpho-word" nếu muốn dùng BARTpho
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_SRC_LEN = 512
MAX_TGT_LEN = 64

def build_input(ex):
    return f"hỏi: {ex['question']} đọc: {ex['passage']}"

def encode(examples):
    inputs = [build_input(ex) for ex in examples]
    targets = [ex["answer_text"] for ex in examples]
    model_inputs = tokenizer(inputs, truncation=True, max_length=MAX_SRC_LEN, padding="max_length")
    labels = tokenizer(targets, truncation=True, max_length=MAX_TGT_LEN, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

C:\Users\ADMIN\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--VietAI--vit5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

TypeError: argument 'vocab': 'dict' object cannot be converted to 'Sequence'

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_dict(encode(train_examples))
dev_ds = Dataset.from_dict(encode(dev_examples))
test_ds = Dataset.from_dict(encode(test_examples))

## 4. Fine-tune mô hình seq2seq

In [ ]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    exact_match = np.mean([p.strip() == l.strip() for p, l in zip(decoded_preds, decoded_labels)])
    scores = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    scores["exact_match"] = exact_match
    return scores

args = Seq2SeqTrainingArguments(
    output_dir="./vimmrc-seq2seq",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=3e-5,
    predict_with_generate=True,
    generation_max_length=MAX_TGT_LEN,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Đánh giá trên tập test

In [ ]:
trainer.evaluate(test_ds)

## 6. Suy luận: đưa (passage, câu hỏi) -> sinh câu trả lời

Bước 4 trong pipeline: nhận `(passage, câu hỏi)` và mô hình **sinh** câu trả lời (không cần danh sách phương án đầu vào).

In [ ]:
def answer_question(passage, question, max_new_tokens=MAX_TGT_LEN):
    text = build_input({"question": question, "passage": passage})
    enc = tokenizer(text, truncation=True, max_length=MAX_SRC_LEN, return_tensors="pt").to(model.device)
    model.eval()
    out_ids = model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

sample = dev_examples[0]
pred_answer = answer_question(sample["passage"], sample["question"])
print("Câu hỏi:", sample["question"])
print("Đáp án dự đoán:", pred_answer)
print("Đáp án đúng:", sample["answer_text"])